# 05 — Trends in AI: Generative AI & Foundation Models — Lab

This lab fine-tunes a pre-trained Hugging Face Transformer checkpoint for two-class sentiment classification. The task demonstrates transfer from large-scale language pre-training to a new supervised task. The final cell reports accuracy and macro-F1 on a bounded, labeled held-out test slice.

## Objectives
- Fine-tune a Hugging Face Transformer checkpoint on a new text classification task and report accuracy and F1 on a held-out test set.

## Prerequisites
- Read the lecture sections on large language models, next-token pre-training, and checkpoint fine-tuning.
- Know Python functions, dictionaries, and basic model evaluation.
- Know the Transformer encoder and attention concepts from m04.

## Required Software and Packages
- Python 3.10 or newer
- `transformers` 4.40 or newer
- `datasets` 2.18 or newer
- `evaluate` 0.4 or newer
- `scikit-learn` 1.3 or newer
- PyTorch 2.1 or newer

## Environment and Setup
Run the next cell in an environment with network access. The model and SST-2 split download from the Hugging Face Hub on first use.

In [ ]:
import numpy as np
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

## Background
A language checkpoint learns token relationships during pre-training. A sequence-classification head maps the checkpoint representation to two sentiment labels. Fine-tuning updates the checkpoint and head together on labeled examples, then evaluation measures predictions on examples withheld from training.

## Exercise Instructions
1. Load `distilbert-base-uncased`, add a two-class sequence-classification head, and tokenize the bounded SST-2 slices. (2 min)
2. Fine-tune the checkpoint with `Trainer` for 3 epochs and inspect the loss reported after each epoch. (6 min)
3. Evaluate the labeled held-out `test` slice and print accuracy and macro-F1. Confirm accuracy reaches at least 0.88. (2 min)

In [ ]:
checkpoint = "distilbert-base-uncased"
dataset = load_dataset(
    "glue",
    "sst2",
    split={"train": "train[:512]", "test": "train[512:640]"},
)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2,
)

def tokenize_batch(batch):
    # TODO: tokenize the sentence field with truncation enabled.
    raise NotImplementedError

In [ ]:
tokenized = dataset.map(tokenize_batch, batched=True)
tokenized = tokenized.rename_column("label", "labels")
tokenized = tokenized.remove_columns(["sentence", "idx"])
tokenized.set_format("torch")

def compute_metrics(prediction):
    predictions = np.argmax(prediction.predictions, axis=-1)
    labels = prediction.label_ids
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
    }

training_args = TrainingArguments(
    output_dir="./m05-sst2-checkpoint",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    evaluation_strategy="epoch",
    logging_strategy="epoch",
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)
trainer.train()
metrics = trainer.evaluate(tokenized["test"])
print(f"accuracy={metrics['eval_accuracy']:.3f}")
print(f"macro_f1={metrics['eval_macro_f1']:.3f}")

## Expected Outputs
The training cell prints one loss record per epoch. The final lines print `accuracy=...` and `macro_f1=...`. Correct training normally produces accuracy at or above `0.880` and macro-F1 near the same value on the bounded SST-2 test slice.

## Questions and Tasks
1. Which operation changes the checkpoint weights: in-context prompting or fine-tuning?
2. Why must the held-out split remain separate from the training split?

In [ ]:
accuracy = metrics["eval_accuracy"]
macro_f1 = metrics["eval_macro_f1"]
assert 0.0 <= accuracy <= 1.0
assert 0.0 <= macro_f1 <= 1.0
assert accuracy >= 0.88, f"accuracy below target: {accuracy:.3f}"
print(f"PASS accuracy={accuracy:.3f}, macro_f1={macro_f1:.3f}")

## Optional
Replace the bounded SST-2 slices with a small equivalent two-class text dataset and keep the same tokenizer, classification head, and metric function. This extension is not required for completion.